# LLaVA-Med 7B 재현 — VQA-RAD (공식 delta 병합) + 프롬프트/보정 탐색

논문: *LLaVA-Med: Training a Large Language-and-Vision Assistant for Biomedicine in One Day* (arXiv:2306.00890).
공개된 VQA-RAD 파인튜닝 delta 를 LLaMA-7B 에 병합해서 평가한다 (직접 파인튜닝하지 않음). **SLAKE 는 별도 노트북**(`LLaVA_Med_SLAKE_QLoRA_reproduction.ipynb`)에 있다 — 공식 SLAKE delta 가 받아지지 않아 QLoRA 근사로 학습한다.

**재현 목표 (Expert-CFG 논문 Table 1, LLaVA-Med VQA-RAD)**: Open 61.52 / Closed 84.19 / Overall 75.19

| 구간 | 내용 |
|---|---|
| 0~3. 환경·베이스·delta·병합 | repo(v1.0.0) + Python 3.10 venv, LLaMA-7B, VQA-RAD delta 병합 → `/content/llava-med-7b-vqarad` |
| 4. 평가 데이터 | test 451문항 (open 179 / closed 272) + 참조 파일 |
| 5. 추론·채점 | 기준 행: closed 전 문항에 `[yes, no]` 접미어, greedy |
| 탐색 (C-16 ~ C-19) | closed 오답 진단, 질문 형태별 접미어, yes/no 편향 보정. **test 기반 탐색이므로 규칙 완화 행으로 따로 표기** |


## 0. 환경 준비

In [18]:
import subprocess, os
subprocess.run('rm -rf /content/LLaVA-Med', shell=True)
subprocess.run(['git', 'clone', 'https://github.com/microsoft/LLaVA-Med.git', '/content/LLaVA-Med'], check=True)
r = subprocess.run(['git', 'checkout', 'v1.0.0'], cwd='/content/LLaVA-Med', capture_output=True, text=True)
print('returncode:', r.returncode); print(r.stderr)
print(subprocess.run(['git', 'describe', '--tags'], cwd='/content/LLaVA-Med', capture_output=True, text=True).stdout)


returncode: 0
Note: switching to 'v1.0.0'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at b9a98a7 fix closed accuracy calculation (#35)

v1.0.0



In [ ]:
# Python 3.10 venv (v1.0.0 의 의존성은 Colab 기본 Python 에서 설치되지 않음). 실패하면 여기서 멈춘다.
# torch 는 2.0.0 으로 고정한다 (고정하지 않으면 -e . 설치 중 최신 torch(cu130)로 올라가 bitsandbytes 0.41 이 CUDA 설정에 실패한다)
!pip install -q uv
!uv venv /content/venv_llava --python 3.10
open('/content/constraints_llava.txt', 'w').write('torch==2.0.0' + chr(10) + 'torchvision==0.15.1' + chr(10))      # torch 고정 파일
!uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt torch==2.0.0 torchvision==0.15.1
!cd /content/LLaVA-Med && uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt -e .
!uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt einops shortuuid ninja open-clip-torch==2.24.0 "numpy<2"
!/content/venv_llava/bin/python -c "import torch, transformers, llava; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"


SyntaxError: unterminated string literal (detected at line 7) (1131144403.py, line 7)

In [14]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os
DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_llavamed'
os.makedirs(DRIVE_ROOT, exist_ok=True)
PY = '/content/venv_llava/bin/python'
print('OK')

Mounted at /content/drive
OK


In [15]:
!uv pip install --python /content/venv_llava/bin/python "numpy<2"
!/content/venv_llava/bin/python -c "import numpy, torch; print(numpy.__version__, torch.__version__); print(torch.from_numpy(numpy.zeros(2)))"


Using Python 3.10.12 environment at: venv_llava
Checked 1 package in 2ms
1.26.4 2.14.1+cu130
tensor([0., 0.], dtype=torch.float64)


## 1. LLaMA-7B 베이스 확보

delta 는 차이값이라 원본 LLaMA-7B(HF 포맷) 위에 더해야 합니다. Drive 에 변환본이 있으면 `LLAMA_BASE` 로 쓰고,
없으면 HF 공개 변환본 `huggyllama/llama-7b` 를 받습니다 (라이선스 확인 후 사용).

In [16]:
import os
from huggingface_hub import snapshot_download

LLAMA_BASE = '/content/drive/MyDrive/llama-7b-hf'
if not os.path.isdir(LLAMA_BASE):
    print(f'{LLAMA_BASE} 없음 → huggyllama/llama-7b 다운로드')
    LLAMA_BASE = snapshot_download('huggyllama/llama-7b', local_dir='/content/llama-7b-hf',
                                   allow_patterns=['*.json', '*.model', '*.safetensors'])
print('LLAMA_BASE =', LLAMA_BASE)
assert os.path.exists(os.path.join(LLAMA_BASE, 'config.json')), 'LLaMA-7B 가중치가 올바르지 않습니다'
assert any(f.endswith(('.safetensors', '.bin')) for f in os.listdir(LLAMA_BASE)), '가중치 파일이 없습니다 (allow_patterns 확인)'

/content/drive/MyDrive/llama-7b-hf 없음 → huggyllama/llama-7b 다운로드


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

LLAMA_BASE = /content/llama-7b-hf


## 2. delta 다운로드

In [19]:
from huggingface_hub import snapshot_download

VQARAD_DELTA = snapshot_download('katielink/llava-med-7b-vqarad-delta', local_dir='/content/delta_vqarad')
SLAKE_DELTA  = snapshot_download('katielink/llava-med-7b-slake-delta',  local_dir='/content/delta_slake')   # 가중치가 없는 빈 저장소(README 만)
print('VQARAD_DELTA =', VQARAD_DELTA)
print('SLAKE_DELTA  =', SLAKE_DELTA)
SLAKE_AVAILABLE = os.path.exists(os.path.join(SLAKE_DELTA, 'config.json'))
print('SLAKE delta 사용 가능:', SLAKE_AVAILABLE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

VQARAD_DELTA = /content/delta_vqarad
SLAKE_DELTA  = /content/delta_slake
SLAKE delta 사용 가능: False


In [25]:
!uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt "transformers @ git+https://github.com/huggingface/transformers@cae78c46" tokenizers==0.12.1 "numpy<2"
!cd /content/LLaVA-Med && /content/venv_llava/bin/python -c "import torch, transformers, llava; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"


Using Python 3.10.12 environment at: /content/venv_llava
Resolved 17 packages in 14.51s                                       
Prepared 2 packages in 2.28s                                             
Uninstalled 2 packages in 45ms
Installed 2 packages in 24msdev0 (from git+https://github.co
 - tokenizers==0.15.2
 + tokenizers==0.12.1
 - transformers==4.36.2
 + transformers==4.28.0.dev0 (from git+https://github.com/huggingface/transformers@cae78c46d658a8e496a815c2ee49b9b178fb9c9a)
/content/venv_llava/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2.0.0+cu117 4.28.0.dev0 True


## 3. delta 병합

venv 의 파이썬으로 실행합니다(인자 이름은 `--base-model-path / --target-model-path / --delta-path`).
병합이 끝나면 delta 는 지워도 됩니다(디스크 절약).

In [26]:
%cd /content/LLaVA-Med
!{PY} -m llava.model.apply_delta \
    --base-model-path {LLAMA_BASE} \
    --target-model-path /content/llava-med-7b-vqarad \
    --delta-path {VQARAD_DELTA}
assert os.path.exists('/content/llava-med-7b-vqarad/config.json'), '병합 실패'

/content/LLaVA-Med
/content/venv_llava/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading base model

Loading delta
/content/venv_llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPVisionModel: ['text_model.encoder.layers.6.mlp.fc1.bias', 'text_model.encoder.layers.2.mlp.fc1.weight', 'text_model.encoder.layers.10.self_attn.out_proj.weight', 'text_model.encoder.layers.2.self_attn.out_proj.weight', 'text_model.encoder.layers.2.self_attn.q_pro

## 4. 평가 데이터 준비

`model_vqa_med.py` 입력: **JSON 배열**, 항목 = `{id, image, conversations:[{from:human,value:질문},{from:gpt,value:정답}]}`.
채점용 참조(`answer`, `answer_type`)는 `*_ref.json` 으로 따로 저장합니다.
VQA-RAD 는 Phi-3.5 노트북과 같은 451문항입니다.
**closed 문항(`answer_type == CLOSED`)에는 질문 뒤에 ` Please choose from the following two options: [yes, no]` 를 붙입니다** (레포 샘플 프롬프트 형식). open 문항은 질문만 보냅니다.


In [28]:
import os, json, shutil, subprocess

EVAL_DIR = '/content/llava_eval_data'
os.makedirs(EVAL_DIR, exist_ok=True)

_GDRIVE_JSON   = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
_GDRIVE_IMGDIR = '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder'

def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

# LLaVA-Med 레포의 closed(yes/no) 문항 프롬프트 형식 (llava/eval/eval_metrics/answer-file.jsonl 의 prompt 필드).
# 파인튜닝 데이터가 이 형식이었다면 접미어 없이 물으면 yes/no 로 답하지 않아 closed 점수가 크게 떨어진다.
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'

def write_eval_files(name, rows, get_image):
    """rows: dict(question, answer, answer_type, ...). 질문 JSON 배열 + 참조 JSON 저장."""
    questions, refs = [], {}
    for i, r in enumerate(rows):
        questions.append({
            'id': i,
            'image': get_image(r),
            'conversations': [{'from': 'human', 'value': r['question'] + (CLOSED_SUFFIX if r['answer_type'] == 'CLOSED' else '')},
                              {'from': 'gpt',   'value': str(r['answer'])}],
        })
        refs[i] = {'answer': str(r['answer']), 'answer_type': r['answer_type']}
    json.dump(questions, open(f'{EVAL_DIR}/{name}_test.json', 'w'))
    json.dump(refs, open(f'{EVAL_DIR}/{name}_ref.json', 'w'))
    n_open = sum(v['answer_type'] == 'OPEN' for v in refs.values())
    print(f'{name}: {len(rows)}문항 (OPEN {n_open} / CLOSED {len(rows) - n_open})')

# ── VQA-RAD ──
_raw_json = _GDRIVE_JSON
if not os.path.exists(_raw_json):
    _raw_json = os.path.join(EVAL_DIR, 'VQA_RAD_Dataset_Public.json')
    subprocess.run(['wget', '-q', '-O', _raw_json, 'https://osf.io/download/9dpze/'], timeout=120, check=True)
all_data = json.load(open(_raw_json))
for item in all_data:
    item['answer_type'] = item.get('answer_type', answer_type_of(item.get('answer', ''))).upper()
_TEST_TYPES = {'test_freeform', 'test_para'}
assert _TEST_TYPES & {str(r.get('phrase_type', '')).lower() for r in all_data}, 'phrase_type 에 test 표시가 없습니다'
vqarad_test = [r for r in all_data if str(r.get('phrase_type', '')).lower() in _TEST_TYPES]
assert len(vqarad_test) == 451, len(vqarad_test)

VQARAD_IMG_DIR = _GDRIVE_IMGDIR
assert os.path.isdir(VQARAD_IMG_DIR), f'이미지 폴더 없음: {VQARAD_IMG_DIR}'
write_eval_files('vqarad', vqarad_test, lambda r: r['image_name'])
assert all(os.path.exists(os.path.join(VQARAD_IMG_DIR, r['image_name'])) for r in vqarad_test), '이미지 누락'

vqarad: 451문항 (OPEN 179 / CLOSED 272)


In [29]:
import json
_qs = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))[:20]
json.dump(_qs, open(f'{EVAL_DIR}/vqarad_smoke.json', 'w'))
%cd /content/LLaVA-Med
!{PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_smoke.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file /content/vqarad_smoke_pred.jsonl
!head -n 5 /content/vqarad_smoke_pred.jsonl | cut -c1-300


/content/LLaVA-Med
/content/venv_llava/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/llava-med-7b-vqarad
/content/venv_llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPVisionModel: ['text_model.encoder.layers.9.layer_norm2.weight', 'text_model.encoder.layers.4.self_attn.out_proj.weight', 'text_model.encoder.layers.10.mlp.fc1.bias', 'text_model.encoder.layers.6.self_attn.q_proj.bias', 'text_model.encoder.layers.5.layer_norm1.weight', '

## 5. 추론 — VQA-RAD

한 문항씩 생성합니다. 질문 파일 앞부분만 잘라 먼저 스모크 테스트를 해보길 권합니다 (오래 걸리면 `--num-chunks`/`--chunk-idx` 로 나눔).

In [15]:
%cd /content/LLaVA-Med
!PYTHONUNBUFFERED=1 nohup {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_test.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {DRIVE_ROOT}/vqarad_pred.jsonl \
    > /content/vqarad_infer.log 2>&1 &


/content


In [85]:
# 1) closed 문항만 분리하면서 접미어를 직접 붙임
import json
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
closed_qs = []
for q in qs:
    if ref[str(q['id'])]['answer_type'] == 'CLOSED':
        v = q['conversations'][0]['value']
        if not v.endswith(CLOSED_SUFFIX):
            q['conversations'][0]['value'] = v + CLOSED_SUFFIX
        closed_qs.append(q)
json.dump(closed_qs, open(f'{EVAL_DIR}/vqarad_closed.json', 'w'))
print(len(closed_qs), '|', closed_qs[0]['conversations'][0]['value'])

# 2) 백그라운드 추론 (기존 open 결과는 건드리지 않음)
%cd /content/LLaVA-Med
!PYTHONUNBUFFERED=1 nohup {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_closed.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {DRIVE_ROOT}/vqarad_pred_closed2.jsonl \
    > /content/vqarad_closed2.log 2>&1 &


272 | Is there evidence of an aortic aneurysm? Please choose from the following two options: [yes, no]
/content


In [ ]:
import time, os
_pred = f'{DRIVE_ROOT}/vqarad_pred_closed2.jsonl'
n = sum(1 for _ in open(_pred)) if os.path.exists(_pred) else 0
print(f'{n}/451 완료')
!tail -c 400 /content/vqarad_infer.log | tr '\r' '\n' | tail -n 3
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader


272/451 완료
100%|██████████| 451/451 [03:40<00:00,  2.04it/s]
0 %, 14876 MiB


In [ ]:
# 3) 끝난 뒤: 기존 open 예측 + 새 closed 예측을 합쳐 채점
import json
old = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred.jsonl')]
new = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_closed2.jsonl')]
assert len(new) == 272, len(new)
new_ids = {p['question_id'] for p in new}
merged = [p for p in old if p['question_id'] not in new_ids] + new
with open(f'{DRIVE_ROOT}/vqarad_pred_v2.jsonl', 'w') as f:
    for p in merged: f.write(json.dumps(p) + '\n')
vqarad_m = score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_v2.jsonl')
print(vqarad_m)


{'open': 64.50784783187018, 'closed': 70.22058823529412, 'overall': 67.95322563615245, 'n_open': 179, 'n_closed': 272}


## 7. 채점

레포의 `run_eval.py` 는 논문 Table 1 의 recall/accuracy 와 정의가 다르므로, Phi-3.5 노트북과 **같은 open recall / closed accuracy 함수**로 직접 채점합니다.

In [104]:
import json, re
import numpy as np
from collections import Counter

def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(gt.strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def score_jsonl(ref_file, pred_file):
    refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
    preds = {}
    for line in open(pred_file):
        p = json.loads(line)
        preds[int(p['question_id'])] = p['text']
    assert set(preds) == set(refs), f'예측 {len(preds)}개 / 참조 {len(refs)}개 — 추론이 끝나지 않았거나 id 불일치'
    cl, op = [], []
    for i, ref in refs.items():
        if ref['answer_type'] == 'CLOSED':
            cl.append(closed_accuracy(preds[i], ref['answer']))
        else:
            op.append(open_recall(preds[i], ref['answer']))
    return {'open': 100 * np.mean(op), 'closed': 100 * np.mean(cl), 'overall': 100 * np.mean(cl + op),
            'n_open': len(op), 'n_closed': len(cl)}

vqarad_m = score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred.jsonl')
_sp = f'{DRIVE_ROOT}/slake_pred.jsonl'
slake_m  = score_jsonl(f'{EVAL_DIR}/slake_ref.json', _sp) if os.path.exists(_sp) else None   # SLAKE delta 없으면 None
print('VQA-RAD:', vqarad_m)
print('SLAKE  :', slake_m)

VQA-RAD: {'open': 64.50784783187018, 'closed': 20.955882352941178, 'overall': 38.241473973181286, 'n_open': 179, 'n_closed': 272}
SLAKE  : None


## 8. 결과 요약

In [108]:
paper = {
    'VQA-RAD': {'open': 61.52, 'closed': 84.19, 'overall': 75.19},
    'SLAKE':   {'open': 83.08, 'closed': 85.34, 'overall': 83.97},
}
ours = {'VQA-RAD': vqarad_m, 'SLAKE': slake_m}

print(f"{'':10s}{'open':>10s}{'closed':>10s}{'overall':>10s}")
for ds in ('VQA-RAD', 'SLAKE'):
    p, o = paper[ds], ours[ds]
    if o is None:
        print(f'[{ds}] 결과 없음'); continue
    print(f'[{ds}]')
    print(f"  paper   {p['open']:10.2f}{p['closed']:10.2f}{p['overall']:10.2f}")
    print(f"  ours    {o['open']:10.2f}{o['closed']:10.2f}{o['overall']:10.2f}")
    print(f"  diff    {o['open']-p['open']:+10.2f}{o['closed']-p['closed']:+10.2f}{o['overall']-p['overall']:+10.2f}")

                open    closed   overall
[VQA-RAD]
  paper        61.52     84.19     75.19
  ours         64.51     70.22     67.95
  diff         +2.99    -13.97     -7.24
[SLAKE] 결과 없음


In [109]:
# 1) 레포 코드를 greedy 로 패치 (원본 문자열이 없으면 assert 로 알림)
import re
p = '/content/LLaVA-Med/llava/eval/model_vqa_med.py'
s = open(p).read()
pat = r"do_sample=True,\s*temperature=0\.7,\s*max_new_tokens=1024,"
if re.search(pat, s):
    s = re.sub(pat, "do_sample=False,\n                max_new_tokens=1024,", s, count=1)
    open(p, 'w').write(s)
    print('greedy 패치 완료')
else:
    assert 'do_sample=False,' in s, '패치 대상 문자열을 찾지 못했습니다'
    print('이미 패치됨')
!grep -n "do_sample" {p}



greedy 패치 완료
235:                do_sample=False,
278:                    do_sample=True,


In [110]:
# 2) 전체 451문항 질문 파일 (closed 에는 접미어) 생성
import json
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
for q in qs:
    v = q['conversations'][0]['value']
    if ref[str(q['id'])]['answer_type'] == 'CLOSED' and not v.endswith(CLOSED_SUFFIX):
        q['conversations'][0]['value'] = v + CLOSED_SUFFIX
json.dump(qs, open(f'{EVAL_DIR}/vqarad_test_suffix.json', 'w'))
print(len(qs), '|', sum(q['conversations'][0]['value'].endswith(CLOSED_SUFFIX) for q in qs), '개 접미어 적용')   # 451 | 272


451 | 272 개 접미어 적용


In [111]:
# 3) greedy 추론 (백그라운드)
%cd /content/LLaVA-Med
!PYTHONUNBUFFERED=1 nohup {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_test_suffix.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {DRIVE_ROOT}/vqarad_pred_greedy.jsonl \
    > /content/vqarad_greedy.log 2>&1 &


/content


In [126]:
import os
_pred = f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
n = sum(1 for _ in open(_pred)) if os.path.exists(_pred) else 0
print(f'{n}/451 완료')
!tail -c 400 /content/vqarad_greedy.log | tr '\r' '\n' | tail -n 3
!pgrep -f model_vqa_med.py > /dev/null && echo "추론 실행 중" || echo "추론 프로세스 없음"


451/451 완료
100%|██████████| 451/451 [03:06<00:00,  2.42it/s]
추론 실행 중


In [127]:
import json, os
from collections import Counter
import numpy as np

_pred = f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
n = sum(1 for _ in open(_pred))
assert n == 451, f'예측 {n}/451 — 추론이 아직 끝나지 않았습니다'

# 프롬프트에 접미어가 실제로 들어갔는지 확인 (closed 는 접미어가 보여야 함)
first = [json.loads(l) for l in open(_pred)][:3]
print('prompt 확인:', [p['prompt'][-70:] for p in first])

greedy_m = score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', _pred)
print(greedy_m)


prompt 확인: ['urysm? Please choose from the following two options: [yes, no]\n<image>', ' side? Please choose from the following two options: [yes, no]\n<image>', 'plane? Please choose from the following two options: [yes, no]\n<image>']
{'open': 66.08140462889067, 'closed': 72.05882352941177, 'overall': 69.68641114982577, 'n_open': 179, 'n_closed': 272}


In [182]:
# 2) 백그라운드 추론 (venv_llava)
import subprocess
_cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_med.py '
        f'--model-name /content/llava-med-7b-vqarad --question-file {EVAL_DIR}/vqarad_closed_alt.json '
        f'--image-folder "{VQARAD_IMG_DIR}" --answers-file {DRIVE_ROOT}/vqarad_pred_closed_altfmt.jsonl')
_log = open('/content/vqarad_closed_alt.log', 'w')
alt_proc = subprocess.Popen(['bash', '-c', _cmd], stdout=_log, stderr=subprocess.STDOUT, start_new_session=True)
print('PID', alt_proc.pid)




PID 56949


In [195]:
import os
_p = f'{DRIVE_ROOT}/vqarad_pred_closed_altfmt.jsonl'
n = sum(1 for _ in open(_p)) if os.path.exists(_p) else 0
print(f'{n}/272 완료 | 프로세스 종료 코드(None=실행 중):', alt_proc.poll())
!tail -n 3 /content/vqarad_closed_alt.log | tr '\r' '\n' | tail -n 2 | cut -c1-200


245/272 완료 | 프로세스 종료 코드(None=실행 중): 1
 90%|█████████ | 245/272 [01:33<00:09,  2.79it/s]


In [202]:
# 3) 끝난 뒤: 기존 greedy open 예측 + 새 closed 예측을 합쳐 채점
old = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')]
new = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_closed_altfmt.jsonl')]
assert len(new) == 272, f'closed 예측 {len(new)}개 — 추론이 아직 끝나지 않았습니다'
new_ids = {p['question_id'] for p in new}
merged = [p for p in old if p['question_id'] not in new_ids] + new
with open(f'{DRIVE_ROOT}/vqarad_pred_altfmt.jsonl', 'w') as f:
    for p in merged: f.write(json.dumps(p) + '\n')
print(score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_altfmt.jsonl'))


{'open': 66.08140462889067, 'closed': 75.0, 'overall': 71.46024707000316, 'n_open': 179, 'n_closed': 272}


## 탐색 — VQA-RAD closed 개선 시도 (C-16 ~ C-19)
위의 기준 추론(`vqarad_pred_greedy.jsonl`)이 있어야 한다. 아래 결과는 test 로 프롬프트·규칙을 고른 탐색이므로 **규칙 준수 행과 별도 행으로 표기**한다.


In [3]:
# C-16. VQA-RAD(공식 delta) closed 예측 진단 — 이 셀만 단독으로 실행해도 된다 (다른 셀 변수에 의존하지 않음).
# 예측이 yes / no / 그 외 중 무엇인지 본다 (프롬프트·디코딩 변형을 고르기 전에 원인 확인).
import json, os, re, glob
from collections import Counter
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/expert_cfg_llavamed')
EVAL_DIR   = globals().get('EVAL_DIR', '/content/llava_eval_data')
def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()
_REF, _PRED = f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
if not os.path.exists(_REF):      # 런타임 초기화로 /content 가 비었을 때: 데이터 셀(C-1)과 같은 방식으로 정답 파일을 다시 만든다
    _src = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
    assert os.path.exists(_src), f'{_src} 가 없습니다'
    _all = json.load(open(_src))
    _test = [r for r in _all if str(r.get('phrase_type', '')).lower() in ('test_freeform', 'test_para')]
    assert len(_test) == 451, len(_test)
    _at = lambda a: 'CLOSED' if str(a).strip().lower() in ('yes', 'no') else 'OPEN'
    os.makedirs(EVAL_DIR, exist_ok=True)
    json.dump({i: {'answer': str(r['answer']), 'answer_type': str(r.get('answer_type', _at(r.get('answer', '')))).upper()}
               for i, r in enumerate(_test)}, open(_REF, 'w'))
    print('정답 파일 재생성:', _REF)
assert os.path.exists(_PRED), f'{_PRED} 가 없습니다 (VQA-RAD greedy 추론 결과)'
refs = {int(k): v for k, v in json.load(open(_REF)).items()}
preds = {}
for _l in open(_PRED):
    _p = json.loads(_l); preds[int(_p['question_id'])] = _p['text']
def cls(text):
    tk = normalize(text); y, n = 'yes' in tk, 'no' in tk
    return 'yes' if y and not n else 'no' if n and not y else 'both' if y and n else 'other'
rows = [(i, refs[i]['answer'].strip().lower(), cls(preds[i]), preds[i]) for i in refs if refs[i]['answer_type'] == 'CLOSED']
yn = [r for r in rows if r[1] in ('yes', 'no')]
print(f'closed {len(rows)}문항 (정답이 yes/no 인 것 {len(yn)}, 아닌 것 {len(rows) - len(yn)})')
print('정답 분포 :', dict(Counter(r[1] if r[1] in ('yes', 'no') else 'other' for r in rows)))
print('예측 분류 :', dict(Counter(r[2] for r in rows)))
print(f'정답이 yes/no 인 {len(yn)}문항 중: 맞음 {sum(r[1] == r[2] for r in yn)} | yes/no 로 답했으나 틀림 {sum(r[2] in ("yes", "no") and r[1] != r[2] for r in yn)} | yes/no 가 아닌 답 {sum(r[2] not in ("yes", "no") for r in yn)}')
print('혼동 (정답→예측):', dict(Counter((r[1], r[2]) for r in yn)))
print('\n예측이 yes/no 로 분류되지 않은 예시 (최대 10):')
for r in [r for r in rows if r[2] in ('other', 'both')][:10]:
    print('  정답', r[1][:20].ljust(20), '←', r[3][:100].replace('\n', ' '))


정답 파일 재생성: /content/llava_eval_data/vqarad_ref.json
closed 272문항 (정답이 yes/no 인 것 251, 아닌 것 21)
정답 분포 : {'yes': 118, 'no': 133, 'other': 21}
예측 분류 : {'no': 101, 'yes': 165, 'other': 6}
정답이 yes/no 인 251문항 중: 맞음 192 | yes/no 로 답했으나 틀림 59 | yes/no 가 아닌 답 0
혼동 (정답→예측): {('yes', 'no'): 9, ('yes', 'yes'): 109, ('no', 'no'): 83, ('no', 'yes'): 50}

예측이 yes/no 로 분류되지 않은 예시 (최대 10):
  정답 left                 ← Assistant: Left side
  정답 supratentorial       ← Assistant: supratentorial
  정답 left                 ← Assistant: Right
  정답 right side           ← Assistant: right side
  정답 both                 ← Assistant: both sides
  정답 flair                ← Assistant: FLAIR


In [31]:
# C-17. VQA-RAD closed 프롬프트 변형 (test 기반 탐색 — 규칙 완화, 별도 행으로 표기할 것).
# 접미어 "[yes, no]" 를 정답 유형이 아니라 '질문 문장'으로 판단해 붙인다: yes/no 형태 질문에만 붙이고, 선택지가 있는 질문("left or right?" 등)은 접미어 없이 원문 그대로 묻는다.
# 이미 있는 greedy 예측(vqarad_pred_greedy.jsonl)을 재사용하고, 접미어를 빼는 문항만 다시 추론하므로 빠르다. 모델 로딩 때문에 1~2분 정도 걸린다(추정).
import json, re, os
from collections import Counter
import numpy as np
if 'score_jsonl' not in globals():      # 채점 함수 셀을 실행하지 않았어도 이 셀만으로 채점되도록 같은 정의를 둔다 (본문 채점 셀과 동일한 방식)
    def normalize(text):
        text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
        return re.sub(r'[^a-z0-9\s]', ' ', text).split()
    def closed_accuracy(pred, gt):
        return float(gt.strip().lower() in normalize(pred))
    def open_recall(pred, gt):
        g, p = Counter(normalize(gt)), Counter(normalize(pred))
        return sum((g & p).values()) / sum(g.values()) if g else 0.0
    def score_jsonl(ref_file, pred_file):
        refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
        preds = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(pred_file)}
        assert set(preds) == set(refs), f'예측 {len(preds)}개 / 참조 {len(refs)}개'
        cl, op = [], []
        for i, r in refs.items():
            (cl if r['answer_type'] == 'CLOSED' else op).append(closed_accuracy(preds[i], r['answer']) if r['answer_type'] == 'CLOSED' else open_recall(preds[i], r['answer']))
        return {'open': 100 * np.mean(op), 'closed': 100 * np.mean(cl), 'overall': 100 * np.mean(cl + op), 'n_open': len(op), 'n_closed': len(cl)}
SUF = ' Please choose from the following two options: [yes, no]'
AUX = ('is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'could', 'has', 'have', 'had', 'will', 'would', 'should')
def is_yesno_form(q):
    q = q.strip().lower()
    return q.split()[0].strip(',') in AUX and not re.search(r'\bor\b', q)

qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
redo = []
for q in qs:
    if ref[str(q['id'])]['answer_type'] != 'CLOSED': continue
    v = q['conversations'][0]['value']
    if v.endswith(SUF): v = v[:-len(SUF)]
    if not is_yesno_form(v):
        q = json.loads(json.dumps(q)); q['conversations'][0]['value'] = v; redo.append(q)
json.dump(redo, open(f'{EVAL_DIR}/vqarad_closed_qform.json', 'w'))
# 분류가 맞았는지 확인용 (정답은 채점·점검에만 사용, 프롬프트 결정에는 쓰지 않음)
_gt = Counter('yes/no' if ref[str(q['id'])]['answer'].strip().lower() in ('yes', 'no') else 'other' for q in redo)
print(f'접미어 없이 다시 묻는 closed 문항: {len(redo)}개 (정답이 yes/no 인 것 {_gt["yes/no"]}, 아닌 것 {_gt["other"]})')
for q in redo[:8]: print('  ', q['conversations'][0]['value'][:90])

# 추론 스크립트가 greedy 가 아니면(do_sample=True, temperature 0.7) 기준 예측(greedy)과 비교가 어긋나므로 먼저 greedy 로 패치한다
_sp = '/content/LLaVA-Med/llava/eval/model_vqa_med.py'
_s = open(_sp).read()
_pat = r"do_sample=True,\s*temperature=0\.7,\s*max_new_tokens=1024,"
if re.search(_pat, _s):
    open(_sp, 'w').write(re.sub(_pat, "do_sample=False,\n                max_new_tokens=1024,", _s, count=1))
    print('추론 스크립트를 greedy 로 패치했습니다')
else:
    print('추론 스크립트: 이미 greedy')
%cd /content/LLaVA-Med
_out = f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl'
if os.path.exists(_out): os.remove(_out)
!PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_closed_qform.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {_out} > /content/vqarad_qform.log 2>&1

new = [json.loads(l) for l in open(_out)]
assert len(new) == len(redo), f'예측 {len(new)}/{len(redo)} — /content/vqarad_qform.log 를 확인하세요'
old = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')]
ids = {p['question_id'] for p in new}
with open(f'{DRIVE_ROOT}/vqarad_pred_qform.jsonl', 'w') as f:
    for p in [p for p in old if p['question_id'] not in ids] + new: f.write(json.dumps(p) + '\n')
print('\n기준(전 문항 접미어, greedy):', score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'))
print('변형(질문 형태 기준 접미어)  :', score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_qform.jsonl'))
print('※ test 기반 탐색 결과입니다. 규칙 준수 행과 별도 행으로 표기하세요.')


In [32]:
# C-18. C-17 에서 접미어를 뺀 문항을 하나씩 본다 — 어떤 질문이었고, 모델이 뭐라고 답했고, 접미어 때문에 바뀐 것은 무엇인지. (단독 실행 가능, 추론 없음)
import json, re, os
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/expert_cfg_llavamed')
EVAL_DIR   = globals().get('EVAL_DIR', '/content/llava_eval_data')
norm = lambda s: re.sub(r'[^a-z0-9\s]', ' ', str(s).lower().replace('<|end|>', ' ').replace('###', ' ')).split()
ref  = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
redo = json.load(open(f'{EVAL_DIR}/vqarad_closed_qform.json'))
ld   = lambda f: {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f)}
base, new = ld(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'), ld(f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl')
ok = lambda pred, gt: gt.strip().lower() in norm(pred)
rows = [(q['id'], q['conversations'][0]['value'], ref[str(q['id'])]['answer']) for q in redo]
print(f'접미어를 뺀 문항 {len(rows)}개 | 기준 정답 {sum(ok(base[i], g) for i, _, g in rows)} → 변형 정답 {sum(ok(new[i], g) for i, _, g in rows)}\n')
for i, qtext, g in rows:
    b, n = ok(base[i], g), ok(new[i], g)
    mark = '↑개선' if (n and not b) else '↓악화' if (b and not n) else ('유지(정답)' if n else '유지(오답)')
    print(f'[{mark}] 정답={g[:25]!r:28} Q: {qtext[:70]}\n         기준→{base[i][:50].replace(chr(10), " ")!r}  |  변형→{new[i][:60].replace(chr(10), " ")!r}')


접미어를 뺀 문항 24개 | 기준 정답 6 → 변형 정답 11

[↑개선] 정답='one'                        Q: Are there multiple or just 1 metastatic focus?
         기준→'Assistant: no'  |  변형→'Assistant: Just one'
[유지(정답)] 정답='Left'                       Q: Is the colon more prominent on the patient's right or left side?
         기준→'Assistant: Left side'  |  변형→'Assistant: Left side'
[유지(오답)] 정답='Smaller'                    Q: Is the heart size in this image smaller or larger than if the image wa
         기준→'Assistant: yes'  |  변형→'Assistant: AP'
[유지(정답)] 정답='supratentorial'             Q: Is this supratentorial or infratentorial?
         기준→'Assistant: supratentorial'  |  변형→'Assistant: Supratentorial'
[유지(오답)] 정답='MRI'                        Q: Is this an MRI or a CT scan?
         기준→'Assistant: no'  |  변형→'Assistant: CT'
[유지(오답)] 정답='Left'                       Q: Is the lesion on the left or right side of the brain?
         기준→'Assistant: Right'  |  변형→'Assistant: Right'
[↑개선] 정답='Female'                     

In [36]:
# C-19. VQA-RAD yes/no 문항의 편향 보정 (test 기반 탐색 — 규칙 완화, 별도 행). 추론 + 분석을 한 셀에서 한다. C-17 을 먼저 (greedy 로) 실행해 둘 것.
# 모델이 "Assistant:" 다음에 내는 첫 답 토큰의 yes/no 로그확률 차이 d = log P(yes) - log P(no) 를 구한다.
# 선택지 순서를 [yes, no] / [no, yes] 로 바꾼 두 번을 물어 평균한다 (순서 편향 완화). 임계값 규칙 3가지를 비교한다:
#   0      : d > 0 이면 yes (보정 없음, 모델 자체 선호)
#   prior  : 예측 yes 비율이 train 의 yes 비율과 같아지도록 임계값을 정한다 (test 정답을 쓰지 않음)
#   oracle : test 정답으로 고른 최적 임계값 (상한 참고용 — 보고하지 말 것)
import json, re, os, subprocess
import numpy as np
from collections import Counter
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/expert_cfg_llavamed')
EVAL_DIR   = globals().get('EVAL_DIR', '/content/llava_eval_data')
PY         = globals().get('PY', '/content/venv_llava/bin/python')
IMG_DIR    = globals().get('VQARAD_IMG_DIR', '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder')
PUBLIC     = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
SUF_A = ' Please choose from the following two options: [yes, no]'
SUF_B = ' Please choose from the following two options: [no, yes]'
AUX = ('is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'could', 'has', 'have', 'had', 'will', 'would', 'should')
def is_yesno_form(q):
    q = q.strip().lower()
    return q.split()[0].strip(',') in AUX and not re.search(r'\bor\b', q)
norm = lambda s: re.sub(r'[^a-z0-9\s]', ' ', str(s).lower().replace('<|end|>', ' ').replace('###', ' ')).split()

# 1) 질문 파일: closed 중 yes/no 형태 문항, 두 가지 선택지 순서 (순서 B 는 id + 100000)
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json')); ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
ynq, base_ids = [], []
for q in qs:
    if ref[str(q['id'])]['answer_type'] != 'CLOSED': continue
    v = q['conversations'][0]['value']
    if v.endswith(SUF_A): v = v[:-len(SUF_A)]
    if not is_yesno_form(v): continue
    base_ids.append(q['id'])
    for k, suf in enumerate((SUF_A, SUF_B)):
        qq = json.loads(json.dumps(q)); qq['id'] = q['id'] + 100000 * k; qq['conversations'][0]['value'] = v + suf; ynq.append(qq)
json.dump(ynq, open(f'{EVAL_DIR}/vqarad_yn2.json', 'w'))
print(f'yes/no 형태 closed 문항 {len(base_ids)}개 x 2가지 순서 = {len(ynq)}회 추론')

# 2) 추론 스크립트 복사본: greedy + 첫 답 토큰의 yes/no 로그확률을 .yn 파일에 기록
_src = open('/content/LLaVA-Med/llava/eval/model_vqa_med.py').read()
_pat = re.compile(r"with torch\.inference_mode\(\):\s*output_ids = model\.generate\(.*?stopping_criteria=\[stopping_criteria\]\)\n", re.S)
_n = len(_pat.findall(_src))
if _n == 0:      # 형태가 다르면 generate 주변을 그대로 보여 준다
    _k = _src.find('model.generate')
    print(_src[max(0, _k - 200):_k + 500] if _k >= 0 else 'model.generate 가 없습니다')
    print('CRLF 줄바꿈:', chr(13) in _src)
assert _n >= 1, 'model_vqa_med.py 의 generate 블록을 찾지 못했습니다 (위 출력을 알려 주세요)'
print(f'generate 블록 {_n}개 발견 -> 첫 번째만 교체')
_new = '''with torch.inference_mode():
            _go = model.generate(input_ids, images=images, do_sample=False, max_new_tokens=24,
                                 stopping_criteria=[stopping_criteria], output_scores=True, return_dict_in_generate=True)
        output_ids = _go.sequences
        _gen = output_ids[0, input_ids.shape[1]:]
        _t = 0
        for _k in range(len(_gen)):      # 'Assistant:' 이후의 첫 토큰 위치를 찾는다
            if ':' in tokenizer.decode(_gen[:_k + 1], skip_special_tokens=True):
                _t = _k + 1
                break
        _vy = [tokenizer.convert_tokens_to_ids(w) for w in ('yes', 'Yes', chr(9601) + 'yes', chr(9601) + 'Yes')]
        _vn = [tokenizer.convert_tokens_to_ids(w) for w in ('no', 'No', chr(9601) + 'no', chr(9601) + 'No')]
        _vy = [x for x in _vy if x is not None and x != tokenizer.unk_token_id]
        _vn = [x for x in _vn if x is not None and x != tokenizer.unk_token_id]
        _ly = _ln = None
        if _t < len(_go.scores):
            _lp = torch.log_softmax(_go.scores[_t][0].float(), -1)
            _ly = torch.logsumexp(_lp[_vy], 0).item()
            _ln = torch.logsumexp(_lp[_vn], 0).item()
        open(args.answers_file + '.yn', 'a').write(json.dumps({'question_id': idx, 'lp_yes': _ly, 'lp_no': _ln, 't': _t,
                                                              'head': tokenizer.decode(_gen[:_t + 3], skip_special_tokens=True)}) + chr(10))
'''
_YN = '/content/LLaVA-Med/llava/eval/model_vqa_yn.py'
open(_YN, 'w').write(_pat.sub(lambda m: _new, _src, count=1))

_out = f'{DRIVE_ROOT}/vqarad_pred_yn2.jsonl'
for f in (_out, _out + '.yn'):
    if os.path.exists(f): os.remove(f)
%cd /content/LLaVA-Med
import time
_cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_yn.py --model-name /content/llava-med-7b-vqarad '
        f'--question-file {EVAL_DIR}/vqarad_yn2.json --image-folder "{IMG_DIR}" --answers-file {_out} > /content/vqarad_yn2.log 2>&1')
_proc = subprocess.Popen(['bash', '-c', _cmd])
_t0 = time.time()
while _proc.poll() is None:      # 진행 상황: 로그확률이 기록된 문항 수 (1분마다 출력)
    _n = sum(1 for _ in open(_out + '.yn')) if os.path.exists(_out + '.yn') else 0
    print(f'[{(time.time() - _t0) / 60:5.1f}분] 진행 {_n}/{len(ynq)}', flush=True)
    time.sleep(60)
assert _proc.returncode == 0, f'추론 실패 (종료 코드 {_proc.returncode}) — /content/vqarad_yn2.log 의 끝부분을 확인하세요'
print(f'추론 완료 ({(time.time() - _t0) / 60:.1f}분)')

# 3) 분석
yn = {}
for l in open(_out + '.yn'):
    r = json.loads(l); yn[r['question_id']] = r
bad = [q['id'] for q in ynq if q['id'] not in yn or yn[q['id']]['lp_yes'] is None]
assert not bad, f'로그확률이 없는 문항 {len(bad)}개 — /content/vqarad_yn2.log 확인. 예: {bad[:5]}'
print('첫 답 위치 확인 (t: 답 토큰 위치, head: 모델 출력 앞부분):')
for q in ynq[:4]: print('  ', yn[q['id']]['t'], repr(yn[q['id']]['head']))
dA = np.array([yn[i]['lp_yes'] - yn[i]['lp_no'] for i in base_ids])
dB = np.array([yn[i + 100000]['lp_yes'] - yn[i + 100000]['lp_no'] for i in base_ids])
gt = np.array([ref[str(i)]['answer'].strip().lower() for i in base_ids])
isyn = np.isin(gt, ['yes', 'no']); gy = (gt == 'yes')

pub = json.load(open(PUBLIC))
_tr = [r for r in pub if str(r.get('phrase_type', '')).lower() not in ('test_freeform', 'test_para') and str(r.get('answer', '')).strip().lower() in ('yes', 'no')]
p_yes = np.mean([str(r['answer']).strip().lower() == 'yes' for r in _tr])
print(f'train 의 yes 비율 {p_yes:.3f} (yes/no 정답 {len(_tr)}개) | 테스트 yes/no 문항 중 정답 yes 비율 {gy[isyn].mean():.3f}')

def acc(pred_yes): return int(((pred_yes == gy) & isyn).sum())          # 정답이 yes/no 가 아닌 문항은 항상 오답
srcs = {'순서A': dA, '순서B': dB, '평균': (dA + dB) / 2}
rules = {}
for name, d in srcs.items():
    cand = np.sort(d)
    best = max(acc(d > (a + b) / 2) for a, b in zip(cand[:-1], cand[1:]))
    rules[name] = {'0': acc(d > 0), 'prior': acc(d > np.quantile(d, 1 - p_yes)), 'oracle(상한)': best}

# 나머지 closed(선택형)는 C-17 의 변형 예측, 기준은 기존 greedy 예측을 쓴다
qf = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl')}
gr = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')}
n_closed = sum(v['answer_type'] == 'CLOSED' for v in ref.values())
other_ok = sum(ref[str(i)]['answer'].strip().lower() in norm(t) for i, t in qf.items())
base_ok = sum(ref[str(i)]['answer'].strip().lower() in norm(gr[i]) for i in base_ids)
print(f'\nyes/no 형태 {len(base_ids)}문항 중 기준(접미어 [yes, no], greedy) 정답: {base_ok}개 (정답이 yes/no 인 문항 {int(isyn.sum())}개)')
print(f'{"입력":6}{"규칙":14}{"yes/no 정답수":>14}{"closed(272)":>14}')
for name, rr in rules.items():
    for rule, n in rr.items():
        print(f'{name:6}{rule:14}{n:>14}{100 * (n + other_ok) / n_closed:>13.2f}%')
print(f'(기준 closed = {100 * (base_ok + other_ok) / n_closed:.2f}%, 선택형 {len(qf)}문항 정답 {other_ok}개 포함)')
print('※ test 기반 탐색입니다. prior 규칙은 test 정답을 쓰지 않지만 선택지 순서·프롬프트를 test 로 골랐으므로 규칙 준수 행과 별도로 표기하세요. oracle 은 상한 참고용이며 보고하지 마세요.')


yes/no 형태 closed 문항 248개 x 2가지 순서 = 496회 추론
generate 블록 2개 발견 -> 첫 번째만 교체
/content/LLaVA-Med
[  0.0분] 진행 0/496
[  1.0분] 진행 85/496
[  2.0분] 진행 256/496
[  3.0분] 진행 426/496
추론 완료 (4.0분)
첫 답 위치 확인 (t: 답 토큰 위치, head: 모델 출력 앞부분):
   3 ' Assistant: no\n##'
   3 ' Assistant: no\n##'
   3 ' Assistant: no\n##'
   3 ' Assistant: no\n##'
train 의 yes 비율 0.498 (yes/no 정답 942개) | 테스트 yes/no 문항 중 정답 yes 비율 0.472

yes/no 형태 248문항 중 기준(접미어 [yes, no], greedy) 정답: 190개 (정답이 yes/no 인 문항 248개)
입력    규칙                yes/no 정답수   closed(272)
순서A   0                        190        73.90%
순서A   prior                    196        76.10%
순서A   oracle(상한)               199        77.21%
순서B   0                        186        72.43%
순서B   prior                    194        75.37%
순서B   oracle(상한)               196        76.10%
평균    0                        188        73.16%
평균    prior                    196        76.10%
평균    oracle(상한)               198        76.84%
(기준 closed = 73.90%, 선택형 24문항 정답 